In [52]:
import pandas as pd
import numpy as np

BASE_DATE = pd.Timestamp("2018-01-01")

exclude_types = [
    'Clubs', 'Muds', 'Hotels', 'Clubs Free',
    "In House Demo's", 'Dealers demo',
    'VIP_CNE', 'Bein Companies PV',
    'Temp', 'Temp OSN'
]

HW_COLS = [
    'Old Decoder Number',
    'New Decoder Number',
    'Old Smartcard Number',
    'New Smartcard Number'
]

In [53]:
bein_original = pd.read_csv(
    r"S:\22.07.26\30596407_BEINDATANEWRPT.CSV",
    dtype="str"
)

swap_original = pd.read_csv(
    r"S:\Sheikh\30592658_SWAPRPT.CSV",
    dtype="str"
)

In [54]:
swap_original["Swap Datetime"] = pd.to_datetime(
    swap_original["Swap Date"].str.split().str[0] + " " + swap_original["Swap Time"],
    format="%d/%m/%Y %I:%M:%S %p"
)

swap = (
    swap_original
    .loc[
        (swap_original["Item"] != "ART Smart Card") &
        (~swap_original["Subscriber Type"].isin(exclude_types))
    ]
    [
        [
            "Swap Datetime",
            "Subscriber Number",
            "Subscriber Type",
            "Replacement Number",
            "Item",
            "Old Serial Number",
            "New Serial Number",
        ]
    ]
    .sort_values(["Subscriber Number", "Swap Datetime"])
)

In [55]:
swap_sc = (
    swap.loc[swap["Item"] == "Smartcard"]
    .rename(
        columns={
            "Old Serial Number": "Old Smartcard Number",
            "New Serial Number": "New Smartcard Number",
        }
    )
    .assign(
        **{
            "Old Decoder Number": pd.NA,
            "New Decoder Number": pd.NA,
        }
    )
)

swap_dec = (
    swap.loc[swap["Item"] == "Decoder"]
    .rename(
        columns={
            "Old Serial Number": "Old Decoder Number",
            "New Serial Number": "New Decoder Number",
        }
    )
    .assign(
        **{
            "Old Smartcard Number": pd.NA,
            "New Smartcard Number": pd.NA,
        }
    )
)

In [56]:
all_swaps = (
    pd.concat([swap_dec, swap_sc], ignore_index=True)
    .sort_values(["Subscriber Number", "Swap Datetime"])
)

In [57]:
bein_base = (
    bein_original[
        [
            "Customer Number",
            "Customer Type",
            "Status",
            "Decoder",
            "Smart Card",
        ]
    ]
    .sort_values('Status')
    .drop_duplicates("Customer Number")
)

bein_base = (
    bein_base.loc[
        ~bein_base["Customer Type"].isin(exclude_types)
    ]
    .rename(
        columns={
            "Customer Number": "Subscriber Number",
            "Customer Type": "Subscriber Type",
            "Decoder": "New Decoder Number",
            "Smart Card": "New Smartcard Number",
        }
    )
)

bein_base["Swap Datetime"] = BASE_DATE

In [58]:
def create_base(df, old_col, new_col, drop_cols):
    base = (
        df.sort_values(["Subscriber Number", "Swap Datetime"])
          .drop_duplicates("Subscriber Number", keep="first")
          .copy()
    )

    base["Swap Datetime"] = BASE_DATE

    return (
        base.drop(columns=drop_cols)
            .rename(columns={old_col: new_col})
    )

In [59]:
base_sc = create_base(
    swap_sc,
    "Old Smartcard Number",
    "New Smartcard Number",
    [
        "Replacement Number",
        "Item",
        "Old Decoder Number",
        "New Smartcard Number",
    ],
)

base_dec = create_base(
    swap_dec,
    "Old Decoder Number",
    "New Decoder Number",
    [
        "Replacement Number",
        "Item",
        "New Decoder Number",
        "Old Smartcard Number",
    ],
)

In [60]:
history = all_swaps.copy()

history[HW_COLS] = (
    history
    .groupby("Subscriber Number")[HW_COLS]
    .ffill()
)

In [61]:
history = (
    history
    .drop(columns=[
        "Replacement Number",
        "Item",
        "Old Smartcard Number",
        "Old Decoder Number",
    ])
    .drop_duplicates()
    .drop_duplicates(
        subset=[
            "Subscriber Number",
            "Swap Datetime",
            "New Decoder Number",
        ],
        keep="last",
    )
    .drop_duplicates(
        subset=[
            "Subscriber Number",
            "Swap Datetime",
            "New Smartcard Number",
        ],
        keep="last",
    )
)

In [62]:
def merge_history(history, base, column):
    history = pd.concat([base, history], ignore_index=True)
    history = history.sort_values(["Subscriber Number", "Swap Datetime"])
    history[column] = (
        history.groupby("Subscriber Number")[column].ffill()
    )
    return history.loc[history["Swap Datetime"] != BASE_DATE]

In [63]:
history = merge_history(
    history,
    base_sc,
    "New Smartcard Number",
)

history = merge_history(
    history,
    base_dec,
    "New Decoder Number",
)

In [64]:
history = pd.concat([bein_base, history], ignore_index=True)
history = history.sort_values(["Subscriber Number", "Swap Datetime"])

history["New Decoder Number"] = (
    history.groupby("Subscriber Number")["New Decoder Number"].ffill()
)

history["New Smartcard Number"] = (
    history.groupby("Subscriber Number")["New Smartcard Number"].ffill()
)

history = history.loc[history["Swap Datetime"] != BASE_DATE]

In [65]:
missing = ~bein_base["Subscriber Number"].isin(history["Subscriber Number"])

history = pd.concat(
    [history, bein_base.loc[missing]],
    ignore_index=True,
)

In [66]:
history = history.loc[(~history['New Smartcard Number'].isna()) & (~history['New Decoder Number'].isna())]


In [67]:
history.loc[history['Subscriber Number']=='10709659']

# history.to_csv('swap timeline.csv', index=False)

,Subscriber Number,Subscriber Type,Status,New Decoder Number,New Smartcard Number,Swap Datetime


In [68]:
history.shape

(841048, 6)